# Query Documentation

## Overview
Well-documented SQL is maintainable, auditable, and understandable by colleagues who didn't write it. Good documentation covers: what the query does, why design choices were made, what the grain is, what NULLs mean, and where the data comes from.

**Documentation layers:**
- **Inline comments** — why, not what; explain non-obvious logic
- **Block headers** — model name, grain, owner, dependencies, update date
- **Data dictionary** — column-level descriptions, types, nullability, example values, tests
- **PostgreSQL COMMENT ON** — server-side metadata queryable from `pg_description`
- **dbt schema.yml descriptions** — version-controlled, tied to models

---

In [1]:
import sqlite3, json
from datetime import date, timedelta

conn = sqlite3.connect(':memory:')
conn.row_factory = sqlite3.Row

conn.executescript("""
-- Mixed dataset: insurance + healthcare + financial transactions
CREATE TABLE customers (
    customer_id  TEXT PRIMARY KEY,
    full_name    TEXT NOT NULL,
    email        TEXT,
    province     TEXT NOT NULL,
    segment      TEXT NOT NULL,   -- 'retail','commercial','group'
    created_at   TEXT NOT NULL
);

CREATE TABLE policies (
    policy_id    TEXT PRIMARY KEY,
    customer_id  TEXT NOT NULL REFERENCES customers(customer_id),
    product_line TEXT NOT NULL,   -- 'auto','home','life','health'
    premium      REAL NOT NULL,
    status       TEXT NOT NULL,   -- 'active','lapsed','cancelled'
    start_date   TEXT NOT NULL,
    end_date     TEXT
);

CREATE TABLE claims (
    claim_id     TEXT PRIMARY KEY,
    policy_id    TEXT NOT NULL REFERENCES policies(policy_id),
    claim_date   TEXT NOT NULL,
    amount       REAL NOT NULL,
    status       TEXT NOT NULL,   -- 'open','approved','denied','paid'
    approved_by  TEXT
);

CREATE TABLE lab_results (
    result_id    TEXT PRIMARY KEY,
    patient_id   TEXT NOT NULL,
    test_name    TEXT NOT NULL,
    result_value REAL,
    unit         TEXT,
    result_date  TEXT NOT NULL,
    flagged      INTEGER DEFAULT 0
);

-- Intentionally seeded with some data quality issues for demos
""")

# Clean customers
customers = [
    ('C001','Alice Nguyen',    'alice@email.com',  'ON','retail',    '2022-01-15'),
    ('C002','Bob Harrington',  'bob@email.com',    'BC','commercial','2022-03-01'),
    ('C003','Fatima Al-Rashid','fatima@email.com', 'QC','group',     '2022-06-10'),
    ('C004','James MacLeod',   'james@email.com',  'NS','retail',    '2023-01-20'),
    ('C005','Mei-Ling Chen',   'mei@email.com',    'AB','commercial','2023-04-05'),
    ('C006','David Park',      None,               'ON','retail',    '2024-01-01'),  # null email
    ('C007','Sandra Okafor',   'sandra@email.com', 'ON','retail',    '2024-02-15'),
]
conn.executemany("INSERT INTO customers VALUES (?,?,?,?,?,?)", customers)

# Policies — including some with data issues
policies = [
    ('POL-001','C001','auto',  1200.0,'active',   '2023-01-15','2024-01-15'),
    ('POL-002','C001','home',  1800.0,'active',   '2023-03-01','2024-03-01'),
    ('POL-003','C002','auto',  2400.0,'active',   '2022-06-01','2023-06-01'),
    ('POL-004','C003','life',   600.0,'lapsed',   '2022-09-01','2023-09-01'),
    ('POL-005','C004','auto',  1400.0,'active',   '2023-02-01','2024-02-01'),
    ('POL-006','C005','health',4800.0,'active',   '2023-05-01','2024-05-01'),
    ('POL-007','C006','home',  2100.0,'cancelled','2024-01-15','2024-06-15'),
    ('POL-008','C007','auto',  1100.0,'active',   '2024-03-01','2025-03-01'),
    ('POL-009','C002','home',  -500.0,'active',   '2024-01-01','2025-01-01'),  # negative premium (data issue)
    ('POL-010','C004','auto',  1400.0,'active',   '2023-02-01','2024-02-01'),  # duplicate of POL-005
]
conn.executemany("INSERT INTO policies VALUES (?,?,?,?,?,?,?)", policies)

# Claims
claims = [
    ('CLM-001','POL-001','2023-06-15',2200.0,'paid',    'Dr. Lee'),
    ('CLM-002','POL-003','2023-08-01', 500.0,'approved','Dr. Pham'),
    ('CLM-003','POL-005','2023-11-20',8500.0,'paid',    'Dr. Singh'),
    ('CLM-004','POL-006','2024-01-10', 350.0,'open',    None),
    ('CLM-005','POL-001','2024-02-28',1100.0,'denied',  'Dr. Lee'),
    ('CLM-006','POL-008','2024-04-15', 750.0,'approved','Dr. Pham'),
    ('CLM-007','POL-003','2024-05-01',99999.0,'open',   None),  # suspiciously large
]
conn.executemany("INSERT INTO claims VALUES (?,?,?,?,?,?)", claims)

# Lab results — with some quality issues
labs = [
    ('LAB-001','P001','HbA1c',    7.2,'%',        '2023-10-01',1),
    ('LAB-002','P001','eGFR',    68.0,'mL/min',   '2023-10-01',0),
    ('LAB-003','P002','HbA1c',    8.4,'%',        '2024-01-10',1),
    ('LAB-004','P002','Creatinine',115.0,'umol/L','2024-01-10',1),
    ('LAB-005','P003','HbA1c',    None,'%',        '2024-02-15',0),  # null value
    ('LAB-006','P001','HbA1c',    7.2,'%',        '2023-10-01',1),  # duplicate of LAB-001
    ('LAB-007','P004','eGFR',   -5.0,'mL/min',   '2024-03-01',0),  # impossible negative
    ('LAB-008','P003','HbA1c',    9.1,'%',        '2024-02-15',1),
]
conn.executemany("INSERT INTO lab_results VALUES (?,?,?,?,?,?,?)", labs)
conn.commit()

print("Testing dataset loaded:")
for tbl in ['customers','policies','claims','lab_results']:
    n = conn.execute(f"SELECT COUNT(*) FROM {tbl}").fetchone()[0]
    print(f"  {tbl:<15s}: {n} rows")
print()
print("Known data quality issues seeded:")
issues = [
    ("customers",    "C006 has NULL email"),
    ("policies",     "POL-009 has negative premium (-500)"),
    ("policies",     "POL-010 is a near-duplicate of POL-005"),
    ("claims",       "CLM-007 has suspiciously large amount (99999)"),
    ("lab_results",  "LAB-005 has NULL result_value"),
    ("lab_results",  "LAB-006 is an exact duplicate of LAB-001"),
    ("lab_results",  "LAB-007 has impossible negative eGFR (-5)"),
]
for tbl, issue in issues:
    print(f"  [{tbl}] {issue}")


Testing dataset loaded:
  customers      : 7 rows
  policies       : 10 rows
  claims         : 7 rows
  lab_results    : 8 rows

Known data quality issues seeded:
  [customers] C006 has NULL email
  [policies] POL-009 has negative premium (-500)
  [policies] POL-010 is a near-duplicate of POL-005
  [claims] CLM-007 has suspiciously large amount (99999)
  [lab_results] LAB-005 has NULL result_value
  [lab_results] LAB-006 is an exact duplicate of LAB-001
  [lab_results] LAB-007 has impossible negative eGFR (-5)


---
## Inline comments and query headers

In [2]:
print("=== SQL inline documentation: comments and naming ===")
print()

well_documented = """
/*
 * model: monthly_loss_ratio
 * purpose: Calculate the insurance loss ratio (claims / premium) by month and product line.
 *          Loss ratio > 1.0 means we paid out more in claims than we collected in premium.
 * grain:   One row per (year, month, product_line)
 * owner:   analytics@company.com
 * updated: 2024-03-01
 * depends: fact_policy, dim_date, dim_product
 */
WITH
-- Step 1: Aggregate premiums by month and product line
monthly_premium AS (
    SELECT
        dd.year,
        dd.month,
        dd.month_name,
        dp.product_line,
        SUM(fp.premium_amount)   AS total_premium,
        COUNT(fp.policy_count)   AS n_policies
    FROM   fact_policy fp
    JOIN   dim_date    dd ON dd.date_key    = fp.date_key
    JOIN   dim_product dp ON dp.product_key = fp.product_key
    WHERE  fp.premium_amount > 0   -- exclude reversals and adjustments
    GROUP  BY dd.year, dd.month, dd.month_name, dp.product_line
),

-- Step 2: Aggregate claims by month and product line
-- Note: claim_date may differ from policy date — we report on when the claim occurred
monthly_claims AS (
    SELECT
        dd.year,
        dd.month,
        dp.product_line,
        SUM(fc.claim_amount) AS total_claims
    FROM   fact_claim  fc
    JOIN   dim_date    dd ON dd.date_key    = fc.date_key
    JOIN   fact_policy fp ON fp.policy_key  = fc.policy_key
    JOIN   dim_product dp ON dp.product_key = fp.product_key
    WHERE  fc.status IN ('approved', 'paid')  -- exclude open/denied claims
    GROUP  BY dd.year, dd.month, dp.product_line
)

-- Final: join and calculate ratio
SELECT
    p.year,
    p.month,
    p.month_name,
    p.product_line,
    p.total_premium,
    COALESCE(c.total_claims, 0)                                      AS total_claims,
    p.n_policies,
    -- Loss ratio: claims / premium. NULL-safe via NULLIF.
    ROUND(COALESCE(c.total_claims, 0) / NULLIF(p.total_premium, 0), 4) AS loss_ratio
FROM   monthly_premium  p
LEFT JOIN monthly_claims c
    ON  c.year         = p.year
    AND c.month        = p.month
    AND c.product_line = p.product_line
ORDER  BY p.year, p.month, p.product_line
"""
print("Well-documented SQL query:")
print(well_documented)


=== SQL inline documentation: comments and naming ===

Well-documented SQL query:

/*
 * model: monthly_loss_ratio
 * purpose: Calculate the insurance loss ratio (claims / premium) by month and product line.
 *          Loss ratio > 1.0 means we paid out more in claims than we collected in premium.
 * grain:   One row per (year, month, product_line)
 * owner:   analytics@company.com
 * updated: 2024-03-01
 * depends: fact_policy, dim_date, dim_product
 */
WITH
-- Step 1: Aggregate premiums by month and product line
monthly_premium AS (
    SELECT
        dd.year,
        dd.month,
        dd.month_name,
        dp.product_line,
        SUM(fp.premium_amount)   AS total_premium,
        COUNT(fp.policy_count)   AS n_policies
    FROM   fact_policy fp
    JOIN   dim_date    dd ON dd.date_key    = fp.date_key
    JOIN   dim_product dp ON dp.product_key = fp.product_key
    WHERE  fp.premium_amount > 0   -- exclude reversals and adjustments
    GROUP  BY dd.year, dd.month, dd.month_name, d

---
## Data dictionary table

In [3]:
print("=== Data dictionary table ===")
print()

# Build a data dictionary table in SQLite
conn.execute("""
    CREATE TABLE IF NOT EXISTS data_dictionary (
        table_name   TEXT NOT NULL,
        column_name  TEXT NOT NULL,
        data_type    TEXT NOT NULL,
        nullable     INTEGER NOT NULL DEFAULT 1,
        description  TEXT,
        example      TEXT,
        tests        TEXT,
        owner        TEXT,
        updated_at   TEXT DEFAULT (date('now')),
        PRIMARY KEY (table_name, column_name)
    )
""")

entries = [
    ('policies','policy_id',    'TEXT',  0,'Unique identifier for each policy. Format: POL-NNNN','POL-001','unique, not_null','data-team','2024-03-01'),
    ('policies','customer_id',  'TEXT',  0,'FK to customers.customer_id. Must exist in customers table.','C001','not_null, relationships(customers)','data-team','2024-03-01'),
    ('policies','product_line', 'TEXT',  0,'Insurance product category.','auto','not_null, accepted_values(auto/home/life/health)','data-team','2024-03-01'),
    ('policies','premium',      'REAL',  0,'Annual premium in CAD. Must be > 0.','1200.00','not_null, not_negative','data-team','2024-03-01'),
    ('policies','status',       'TEXT',  0,'Current policy lifecycle status.','active','accepted_values(active/lapsed/cancelled)','data-team','2024-03-01'),
    ('policies','start_date',   'TEXT',  0,'Policy effective date. ISO 8601 (YYYY-MM-DD).','2024-01-15','not_null','data-team','2024-03-01'),
    ('policies','end_date',     'TEXT',  1,'Policy expiry date. NULL for open-ended policies.','2025-01-15','end_date > start_date','data-team','2024-03-01'),
    ('claims','claim_id',       'TEXT',  0,'Unique claim identifier. Format: CLM-NNN.','CLM-001','unique, not_null','data-team','2024-03-01'),
    ('claims','amount',         'REAL',  0,'Claimed amount in CAD. Must be positive.','2200.00','not_null, not_negative, < 500000','data-team','2024-03-01'),
    ('claims','status',         'TEXT',  0,'Current claim status.','paid','accepted_values(open/approved/denied/paid)','data-team','2024-03-01'),
    ('lab_results','result_value','REAL',1,'Numeric lab result. NULL for results awaiting processing.','7.2','accepted_range(0,30) when test=HbA1c','clinical-team','2024-03-01'),
    ('lab_results','flagged',   'INTEGER',0,'1 if result is outside reference range; 0 if normal.','1','accepted_values(0,1)','clinical-team','2024-03-01'),
]
conn.executemany("INSERT OR REPLACE INTO data_dictionary VALUES (?,?,?,?,?,?,?,?,?)", entries)
conn.commit()

rows = conn.execute("""
    SELECT table_name, column_name, data_type,
           CASE nullable WHEN 0 THEN 'NOT NULL' ELSE 'nullable' END AS nullable,
           tests
    FROM   data_dictionary
    ORDER  BY table_name, column_name
""").fetchall()

print(f"  {'table':<14s}  {'column':<14s}  {'type':<8s}  {'nullable':<9s}  tests")
print("  " + "-"*75)
for r in rows:
    print(f"  {r['table_name']:<14s}  {r['column_name']:<14s}  {r['data_type']:<8s}  {r['nullable']:<9s}  {r['tests']}")

print()
print("PostgreSQL: use pg_catalog for auto-generated data dictionary:")
print("""
  SELECT
      c.table_name,
      c.column_name,
      c.data_type,
      c.is_nullable,
      c.column_default,
      pgd.description     AS column_comment
  FROM   information_schema.columns c
  LEFT JOIN pg_catalog.pg_statio_all_tables st
         ON st.schemaname = c.table_schema
         AND st.relname   = c.table_name
  LEFT JOIN pg_catalog.pg_description pgd
         ON pgd.objoid    = st.relid
         AND pgd.objsubid = c.ordinal_position
  WHERE  c.table_schema = 'public'
  ORDER  BY c.table_name, c.ordinal_position;

  -- Add column comments (visible in pg_description):
  COMMENT ON COLUMN policies.premium IS 'Annual premium in CAD. Must be positive.';
""")


=== Data dictionary table ===

  table           column          type      nullable   tests
  ---------------------------------------------------------------------------
  claims          amount          REAL      NOT NULL   not_null, not_negative, < 500000
  claims          claim_id        TEXT      NOT NULL   unique, not_null
  claims          status          TEXT      NOT NULL   accepted_values(open/approved/denied/paid)
  lab_results     flagged         INTEGER   NOT NULL   accepted_values(0,1)
  lab_results     result_value    REAL      nullable   accepted_range(0,30) when test=HbA1c
  policies        customer_id     TEXT      NOT NULL   not_null, relationships(customers)
  policies        end_date        TEXT      nullable   end_date > start_date
  policies        policy_id       TEXT      NOT NULL   unique, not_null
  policies        premium         REAL      NOT NULL   not_null, not_negative
  policies        product_line    TEXT      NOT NULL   not_null, accepted_values(auto/h

---
## Common Pitfalls

**1. Comments that describe what, not why**
`-- count the rows` above `SELECT COUNT(*)` adds no value — the code already says that. Useful comments explain *why*: `-- exclude reversals (negative premiums from system corrections, not real policies)` or `-- claim_date not policy_date: we report when the claim occurred, not when the policy was issued`.

**2. Data dictionary not kept in sync with schema**
A data dictionary that describes columns which no longer exist, or omits new ones, is worse than no dictionary — it actively misleads. Automate dictionary generation from `information_schema.columns` + `pg_description` (PostgreSQL COMMENT ON) so it stays current with the actual schema.

**3. No description of the grain in model documentation**
The single most important thing to document for any table or view is its grain: 'one row per policy' vs 'one row per policy per month'. Without this, consumers cannot know whether aggregating the table will double-count.

**4. Undocumented NULLs**
A NULL in `end_date` could mean 'open-ended policy', 'unknown expiry', or 'data not yet loaded'. These have very different business meanings. Always document what NULL means for every nullable column.

**5. Losing documentation when renaming or dropping columns**
In PostgreSQL, `ALTER TABLE t RENAME COLUMN a TO b` does not preserve the COMMENT ON the old column. Re-apply comments after any column rename. In dbt, schema.yml column documentation survives renames as long as the YAML is updated.

**6. No lineage documentation (what feeds what)**
A model's description should state its upstream dependencies and downstream consumers. dbt's `ref()` graph provides automatic lineage, but a brief `depends: [fact_policy, dim_date]` and `used_by: [monthly_dashboard, regulatory_report]` in the model description saves hours of investigation when debugging a broken pipeline.


---
*sql_methods_library - Samantha McGarrigle*